# Equação de Poisson 1D — Resolução Método Galerkin-Colocação

## 1. Características da Equação de Poisson

- **Linearidade**:
A equação $\partial_x^2 u = f(x)$ é linear, com coeficientes constantes e termo fonte $f$ desacoplado de $u$. O sistema resultante é uma única equação algébrica linear.

- **Ausência de singularidade na origem**:
Não há termos do tipo $1/r$ ou $1/r^2$. O domínio é simplesmente $[a,b]$, e a regularidade da solução é determinada apenas por $f$ e pelas condições de contorno.

- **Domínio fixo**:
O comprimento $L = b-a$ é um parâmetro de entrada, não uma incógnita.

- **Sem rigidez intrínseca**:
Se $f$ for suave, a solução é suave.

## 2. Cuidados ao Aplicar o Método Pseudo Espectral

- **Mapeamento de domínio**:
Chebyshev é definido em $[-1,1]$; aplicamos $x_j = \frac{a+b}{2} + \frac{b-a}{2}\xi_j$.

- **Matriz de diferenciação**:
Usamos $D^{(2)} = D \cdot D$ (produto matricial), não a fórmula analítica explícita, para preservar a precisão em $N$ moderado.

- **Imposição das condições de contorno**:
Substituímos as linhas $0$ e $N$ do sistema $D^{(2)}u = f$ pelas condições de Dirichlet. Isso destrói a estrutura espectral nessas duas linhas, mas é adequado para problemas suaves.

- **Pontos de colocação GLC**:
Existe uma densidade maior nas bordas, o que é vantajoso para capturar gradientes próximos a $a$ e $b$.

- **Suavidade da fonte $f$**:
A convergência espectral exige $f$ analítica. Descontinuidades em $f$ degradam a convergência de exponencial para algébrica e podem gerar oscilações na evolução do erro numérico.

## 3. Implicações para a Estrutura da Solução

### 3.1 Paridade na origem (caso $a=0$)

Se o domínio for $[0, b]$ e $f$ for **par** em $x=0$ (tipicamente $f(x) = f_0 + f_2 x^2 + \dots$), então:

- $u(x)$ é **par**: $u(x) = u_0 + u_2 x^2 + u_4 x^4 + \dots$
- $\partial_x u$ é **ímpar**: $\partial_x u(0) = 0$
- A condição de regularidade em $x=0$ é naturalmente satisfeita por uma base com apenas potências pares

Nesse caso, usamos **apenas polinômios de Chebyshev de ordem par** $T_{2k}$, o que melhora a convergência perto da origem.

Se $f$ for **ímpar** em $x=0$, a solução também é ímpar, e usamos apenas $T_{2k+1}$.

### 3.2 Caso geral em $[a,b]$

Como não há restrição de paridade usamos toda a base $T_0, \dots, T_N$.

## 4. Tabela Resumo

| Aspecto | Equação de Poisson 1D |
|---|---|
| Natureza | Linear, desacoplada |
| Singularidade na origem | Nenhuma |
| Domínio | Fixo $[a,b]$ |
| Regularidade em $x=0$ | Paridade de $f$ |
| Polinômios de Chebyshev | Completa |
| Condições de contorno | Dirichlet por substituição de [a,b] |

## 5. Conclusão

A equação de Poisson 1D é um caso simples para aplicação do método pseudo-espectral:

1. **Sem singularidades** — nenhuma regularização necessária.
2. **Domínio fixo** — sem autovalor de fronteira livre.
3. **Linear** — resolvemos com uma única fatoração LU.
4. **Paridade opcional** — restrição de base só se $f$ tiver simetria definida em $x=0$.

O único cuidado real é a **suavidade de $f$** (para manter a convergência exponencial do erro). Fora isso, o método é direto: montamos $D^{(2)}$, impomos as condições de contorno, resolvemos o sistema e validamos comparando com alguma solução analítica de exemplo.

# Equação de Poisson 3D em Eletrostática — Resolução Método Galerkin-Colocação

## 1. A Equação de Poisson em Eletrostática

### 1.1 Formulação física

Em eletrostática, o potencial elétrico $\phi(\mathbf{x})$ gerado por uma distribuição de carga $\rho(\mathbf{x})$ satisfaz a equação de Poisson:

$$
\boxed{\nabla^2 \phi(\mathbf{x}) = -\frac{\rho(\mathbf{x})}{\varepsilon_0}}
$$

onde:
- $\phi(\mathbf{x})$ é o **potencial elétrico** (em Volts)
- $\rho(\mathbf{x})$ é a **densidade volumétrica de carga** (em C/m³)
- $\varepsilon_0 \approx 8.854 \times 10^{-12}$ F/m é a **permissividade do vácuo**

### 1.2 Interpretação física

- **Linearidade**: o potencial é linear na densidade de carga, pelo princípio da superposição.
- **Ausência de singularidade no interior**: no interior do domínio, $\phi$ é suave se $\rho$ for suave. Singularidades só aparecem quando $\rho$ é uma delta de Dirac (carga puntual).
- **Domínio fixo**: consideramos $\Omega = [a_x, b_x] \times [a_y, b_y] \times [a_z, b_z]$, com $a_x = a_y = a_z = -1$ e $b_x = b_y = b_z = 1$ neste desenvolvimento.
- **Condições de contorno**: nas faces do domínio, o potencial satisfaz as condições de contorno de Dirichlet, por exemplo, as faces são condutores aterrados ($\phi = 0$) ou mantidos em um potencial fixo.

### 1.3 Significado da fonte

Seja $\nabla^2 u = f$ com

$$
f(\mathbf{x}) = -\frac{\rho(\mathbf{x})}{\varepsilon_0}.
$$

A escolha que reproduz o caso do notebook corresponde a uma densidade de carga proporcional a uma autofunção do Laplaciano no cubo $[-1,1]^3$, que se anula em todas as faces.

## 2. Dedução Analítica por Separação de Variáveis

### 2.1 A equação

Resolvemos

$$
\nabla^2 \phi(x,y,z) = -\frac{\rho(x,y,z)}{\varepsilon_0}, \qquad (x,y,z) \in \Omega = [-1,1]^3,
$$

com condições de contorno de Dirichlet homogêneas:

$$
\phi = 0 \quad \text{em } \partial\Omega \ \text{(faces aterradas)}.
$$

### 2.2 Autofunções do Laplaciano no cubo $[-1,1]^3$

As autofunções do operador $-\nabla^2$ no cubo $[-1,1]^3$ com condições de Dirichlet homogêneas são:

$$
\boxed{
\psi_{nml}(x,y,z) = \sin\!\left(\frac{n\pi(x+1)}{2}\right)
\sin\!\left(\frac{m\pi(y+1)}{2}\right)
\sin\!\left(\frac{l\pi(z+1)}{2}\right),
\qquad n,m,l \in \mathbb{N}^*
}
$$

com autovalores

$$
\boxed{
\Lambda_{nml} = \frac{\pi^2}{4}(n^2 + m^2 + l^2)
}
$$

Essas funções formam uma base completa de $L^2(\Omega)$, ou seja, qualquer função suave que se anula nas faces pode ser expandida nessa base. Note que a normalização por $(x+1)/2$ mapeia $[-1,1]$ para $[0,1]$, e a autofunção se anula em $x = \pm 1$, $y = \pm 1$, $z = \pm 1$.

### 2.3 Expansão do potencial e da carga

Expandimos o potencial e a densidade de carga:

$$
\phi = \sum_{n,m,l} \Phi_{nml}\,\psi_{nml},
\qquad
\rho = \sum_{n,m,l} R_{nml}\,\psi_{nml},
$$

onde os coeficientes da carga são as projeções

$$
R_{nml} = \int_{-1}^{1}\!\!\int_{-1}^{1}\!\!\int_{-1}^{1}
\rho(x,y,z)\,\psi_{nml}(x,y,z)\,dx\,dy\,dz
\;\Big/\; \|\psi_{nml}\|^2,
$$

sendo $\|\psi_{nml}\|^2 = 1$ para a base ortonormalizada (ou $1$ para a base não normalizada, conforme a convenção).

Substituindo em $\nabla^2 \phi = -\rho/\varepsilon_0$:

$$
\sum_{n,m,l} \Phi_{nml}\,(-\Lambda_{nml})\,\psi_{nml}
= -\frac{1}{\varepsilon_0}\sum_{n,m,l} R_{nml}\,\psi_{nml}.
$$

Pela independência linear das autofunções, igualamos coeficiente a coeficiente:

$$
\boxed{
\Phi_{nml} = \frac{R_{nml}}{\varepsilon_0\,\Lambda_{nml}}
= \frac{4\,R_{nml}}{\varepsilon_0\,\pi^2(n^2+m^2+l^2)}
}
$$

O Laplaciano age diagonalmente na base de autofunções e o potencial é obtido dividindo o coeficiente de carga pelo autovalor correspondente.

### 2.4 Solução geral em série

$$
\boxed{
\phi(x,y,z) = \frac{1}{\varepsilon_0}\sum_{n,m,l=1}^{\infty}
\frac{4\,R_{nml}}{\pi^2(n^2+m^2+l^2)}
\sin\!\left(\frac{n\pi(x+1)}{2}\right)
\sin\!\left(\frac{m\pi(y+1)}{2}\right)
\sin\!\left(\frac{l\pi(z+1)}{2}\right)
}
$$

Que é o análogo 3D da série de Fourier senoidal para o problema eletrostático de Dirichlet no cubo $[-1,1]^3$.

## 3. Exemplo a ser Implementado

### 3.1 Densidade de carga

No código, escolhemos uma densidade de carga proporcional à autofunção fundamental no domínio $[-1,1]^3$:

$$
\rho(x,y,z) = A \sin\!\left(\frac{\pi(x+1)}{2}\right)
\sin\!\left(\frac{\pi(y+1)}{2}\right)
\sin\!\left(\frac{\pi(z+1)}{2}\right),
$$

com $A = \frac{3\pi^2}{4}\varepsilon_0$. Comparando com a expansão em autofunções, vemos que apenas o modo $(n,m,l) = (1,1,1)$ é não-nulo:

$$
R_{111} = \frac{3\pi^2}{4}\varepsilon_0, \qquad R_{nml} = 0 \ \text{caso contrário}.
$$

A carga é exatamente uma autofunção do Laplaciano no domínio $[-1,1]^3$.

### 3.2 Potencial resultante

Como $\Lambda_{111} = \frac{\pi^2}{4}(1+1+1) = \frac{3\pi^2}{4}$:

$$
\Phi_{111} = \frac{R_{111}}{\varepsilon_0\,\Lambda_{111}}
= \frac{\frac{3\pi^2}{4}\varepsilon_0}{\varepsilon_0 \cdot \frac{3\pi^2}{4}} = 1.
$$

Portanto, o potencial elétrico é

$$
\boxed{
\phi(x,y,z) = \sin\!\left(\frac{\pi(x+1)}{2}\right)
\sin\!\left(\frac{\pi(y+1)}{2}\right)
\sin\!\left(\frac{\pi(z+1)}{2}\right)
}
$$

Verificação:

$$
\nabla^2 \phi = -\frac{3\pi^2}{4} \sin\!\left(\frac{\pi(x+1)}{2}\right)
\sin\!\left(\frac{\pi(y+1)}{2}\right)
\sin\!\left(\frac{\pi(z+1)}{2}\right) = -\frac{\rho}{\varepsilon_0},
\qquad \phi|_{\partial\Omega} = 0.
$$

### 3.3 Interpretação física

- **Modo fundamental**: o potencial é uma única autofunção, o modo $(1,1,1)$ no domínio $[-1,1]^3$. Este modo satisfaz as condições de contorno com um único pico positivo no centro do cubo (em $(0,0,0)$) e zeros nas seis faces ($x=\pm 1$, $y=\pm 1$, $z=\pm 1$).
- **Simetria**: $\phi$ é simétrica sob permutação de $x,y,z$ e sob reflexão $x \to -x$, etc., herança da simetria do cubo e da distribuição de carga.
- **Sinal**: o potencial é positivo no centro, o que está de acordo com uma carga positiva concentrada na origem.
- **Amplitude**: o potencial varia entre $-1$ V e $+1$ V no domínio.

## 4. Cuidados ao Aplicar o Método Pseudo-Espectral

- **Mapeamento de domínio**:
Chebyshev é definido em $[-1,1]$; aplicamos em cada direção
$$
x_j = \frac{a_x+b_x}{2} + \frac{b_x-a_x}{2}\xi_j, \quad
y_k = \frac{a_y+b_y}{2} + \frac{b_y-a_y}{2}\xi_k, \quad
z_\ell = \frac{a_z+b_z}{2} + \frac{b_z-a_z}{2}\xi_\ell.
$$
Com $a=-1$ e $b=1$, o mapeamento reduz-se à identidade em cada direção, e os pontos de colocação coincidem com os pontos GLC de Chebyshev no domínio físico.

- **Matriz de diferenciação**:
Usamos $D_x^{(2)} = D_x \cdot D_x$ (produto matricial), e analogamente para $y$ e $z$, para preservar a precisão em $N$ moderado.

- **Operador Laplaciano discreto**:
O Laplaciano 3D em malha tensorial é construído via **soma de Kronecker**:
$$
\mathcal{L} = D_x^{(2)} \otimes I_y \otimes I_z + I_x \otimes D_y^{(2)} \otimes I_z + I_x \otimes I_y \otimes D_z^{(2)}.
$$
Isso evita construir manualmente matrizes 3D densas e preserva a estrutura tensorial.

- **Imposição das condições de contorno**:
Substituímos as linhas correspondentes aos pontos de fronteira do sistema $\mathcal{L}\phi = f$ pelas condições de Dirichlet ($\phi_i = 0$, faces aterradas em $x=\pm 1$, $y=\pm 1$, $z=\pm 1$). Isso destrói a estrutura espectral nessas linhas, mas é adequado para problemas suaves.

- **Pontos de colocação GLC**:
Em cada direção, há densidade maior nas bordas, o que é vantajoso para capturar gradientes próximos às faces do paralelepípedo.

- **Suavidade da fonte $\rho$**:
A convergência espectral exige $\rho$ analítica. Descontinuidades na densidade de carga degradam a convergência de exponencial para algébrica e podem gerar oscilações.

- **Custo computacional**:
A matriz $\mathcal{L}$ tem tamanho $(N_x+1)(N_y+1)(N_z+1) \times \cdots$, ou seja, cresce rápido com $N$. Para $N=20$ em cada direção, são $\sim 9000$ incógnitas; para $N=40$, $\sim 69000$. O custo de LU denso é $\mathcal{O}(M^3)$ com $M$ o número total de incógnitas, o que mostra um rápido crescimento das operações a serem processadas.

## 5. Implicações para a Estrutura da Solução

### 5.1 Paridade nas faces (caso de domínio simétrico)

Se o domínio for simétrico em relação a um plano, por exemplo $x=0$, e $\rho$ for **par** em $x$, então:

- $\phi(\mathbf{x})$ é **par** em $x$: $\phi(x,y,z) = \phi(-x,y,z)$
- $\partial_x \phi$ é **ímpar** em $x$: $\partial_x \phi(0,y,z) = 0$
- A condição de regularidade em $x=0$ é naturalmente satisfeita por uma base com apenas potências pares em $x$

Nesse caso, usamos **apenas polinômios de Chebyshev de ordem par** $T_{2k}$ na direção $x$, o que melhora a convergência perto do plano de simetria.

### 5.2 Caso geral em $\Omega = [a_x,b_x]\times[a_y,b_y]\times[a_z,b_z]$

Como não há restrição de paridade, usamos toda a base $T_0, \dots, T_{N_x}$ em cada direção, e analogamente para $y$ e $z$.

### 5.3 Condições de contorno

Para condições de contorno de Dirichlet homogêneas ($\phi=0$ em $\partial\Omega$), as linhas da matriz correspondentes aos pontos de fronteira são substituídas por $\phi_i = 0$. Para condições de contorno de Dirichlet não-homogêneas, $\phi_i = V(\mathbf{x}_i)$ diferente de zero, o potencial pode ser prescrito nas faces (condutores mantidos a potenciais fixos).

## 6. Conexão entre a Dedução Analítica e a Formulação Espectral

No código, resolvemos exatamente o mesmo problema da dedução analítica, mas em vez de expandir em autofunções analíticas (que só existem para geometrias simples), expandimos em polinômios de Chebyshev e impomos as condições de contorno por substituição direta dos valores nas linhas da matriz.

| Elemento | Dedução analítica | Código espectral |
|---|---|---|
| Base de expansão | $\sin\!\left(\frac{n\pi(x+1)}{2}\right)\sin\!\left(\frac{m\pi(y+1)}{2}\right)\sin\!\left(\frac{l\pi(z+1)}{2}\right)$ | $T_k(x)T_l(y)T_m(z)$ |
| Operador | $\Lambda_{nml} = \frac{\pi^2}{4}(n^2+m^2+l^2)$ | $\mathcal{L}$ (Kronecker de $D^{(2)}$) |
| Coeficientes do potencial | $\Phi_{nml} = R_{nml}/(\varepsilon_0\Lambda_{nml})$ | $\phi = \mathcal{L}^{-1} b$ |
| Condições de Contorno | automáticas pela base | impostas por substituição direta |
| Aplicabilidade | geometrias simples | geometrias e fontes arbitrárias |

Ambos os caminhos devem levar ao mesmo potencial elétrico:

$$
\phi(x,y,z) = \sin\!\left(\frac{\pi(x+1)}{2}\right)
\sin\!\left(\frac{\pi(y+1)}{2}\right)
\sin\!\left(\frac{\pi(z+1)}{2}\right),
$$

com erro da máquina ($\sim 10^{-14}$) quando $N$ é suficientemente grande. A diferença é que o método espectral numérico generaliza para distribuições de carga arbitrárias e geometrias não-triviais, onde a expansão analítica em autofunções não tem forma fechada.

## 7. Tabela Resumo

| Aspecto | Equação de Poisson 3D em Eletrostática |
|---|---|
| Equação | $\nabla^2 \phi = -\rho/\varepsilon_0$ |
| Natureza | Linear, desacoplada |
| Singularidade no interior | Nenhuma para $\rho$ suave |
| Domínio | Fixo $\Omega = [-1,1]^3$ |
| Condições de contorno | Dirichlet ($\phi=0$, faces aterradas) |
| Autofunções analíticas | $\psi_{nml} = \sin\!\left(\frac{n\pi(x+1)}{2}\right)\sin\!\left(\frac{m\pi(y+1)}{2}\right)\sin\!\left(\frac{l\pi(z+1)}{2}\right)$ |
| Autovalores | $\Lambda_{nml} = \frac{\pi^2}{4}(n^2+m^2+l^2)$ |
| Coeficientes do potencial | $\Phi_{nml} = R_{nml}/(\varepsilon_0\Lambda_{nml})$ |
| Exemplo de implementação | $\rho = \frac{3\pi^2}{4}\varepsilon_0\psi_{111} \Rightarrow \Phi_{111} = 1$ |
| Potencial analítico | $\phi = \sin\!\left(\frac{\pi(x+1)}{2}\right)\sin\!\left(\frac{\pi(y+1)}{2}\right)\sin\!\left(\frac{\pi(z+1)}{2}\right)$ |
| Polinômios de Chebyshev | Expansão em cada direção |

## 8. Conclusão

A equação de Poisson 3D em eletrostática é uma aplicação direta do método espectral:

1. **Sem singularidades:** para densidades de carga suaves, o potencial é suave.
2. **Domínio fixo:** sem autovalor de fronteira livre.
3. **Linear:** resolvemos com uma única fatoração LU.
4. **Estrutura tensorial:** o Laplaciano é construído por soma de Kronecker.
5. **Paridade opcional:** restrição de base só se $\rho$ tiver simetria definida em relação a algum plano.
6. **Dedução analítica via separação de variáveis:** com o código devemos obter o potencial fechado $\phi = \sin\!\left(\frac{\pi(x+1)}{2}\right)\sin\!\left(\frac{\pi(y+1)}{2}\right)\sin\!\left(\frac{\pi(z+1)}{2}\right)$, que deve concordar com a solução geral senoidal em série de Fourier para $\rho$ arbitrária.

Os maiores cuidados são a suavidade de $\rho$ (para manter a convergência exponencial do erro) e o custo computacional (que cresce como $N^3$ em memória e $N^6$ em tempo de processamento para operações matriciais). Fora isso, o método é direto: montamos o Laplaciano tensorial, impomos as condições de contorno, resolvemos o sistema e validamos comparando com o potencial analítico exato.

**Observação sobre o domínio $[-1,1]^3$:** todos os resultados acima — autofunções, autovalores, coeficientes e expressão final do potencial — refletem diretamente o novo domínio. A escolha $A = \frac{3\pi^2}{4}\varepsilon_0$ é o que garante que o potencial resultante tenha amplitude unitária ($\pm 1$ V) no cubo $[-1,1]^3$, em coerência com o que o pipeline numérico está produzindo.